# Comp Lab 1 — Clinical Variant Interpretation
**BIO 462/594 Molecular Biology | Tuesday, September 29 | OSC OnDemand (Pitzer)**

You have a list of DNA variants in three human genes. Today you will:

1. **Score** every possible missense variant in **TP53**, **BRCA1**, and **RPS19** using published **AlphaMissense** predictions.
2. **Reconcile** those scores against **ClinVar**, the public archive of what clinical laboratories actually assert about each variant.
3. **Apply ACMG criteria** — the rules clinical geneticists use to turn evidence into a classification — and decide where you agree or disagree with ClinVar.

**Why these three genes?** All three cause inherited disease, but their biology is completely different:
- **TP53** — tumor suppressor; somatic hotspots in cancer; dominant-negative mechanism (Ch. 5)
- **BRCA1** — DNA repair; heterozygous carriers; loss of heterozygosity (Ch. 5)
- **RPS19** — ribosomal protein; recessive; Diamond-Blackfan anemia (Ch. 11)

Same tool, same rules — three different variant landscapes. That contrast is the point of today's lab.

## How this notebook works
- Run cells top to bottom with **Shift+Enter**. Almost everything is written for you — **no coding background needed**.
- Cells marked **Your turn** ask you to change one thing (a gene name, a threshold) or type a short answer in a text cell.
- If a cell errors, first check that earlier cells ran, then ask. Nothing you do here can break anything permanently — Kernel → Restart & Run All always resets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'
DATA = 'data'   # folder holding the lab files

def load_gene(gene):
    df = pd.read_csv(f'{DATA}/{gene}_alphamissense_clinvar.tsv', sep='\t')
    return df

print('ready')

---
## Part 1 — Meet the data (10 min)

Each gene file contains **every missense substitution AlphaMissense scored** on that gene's protein, joined to what **ClinVar** says about variants that have been seen in patients.

| column | meaning |
|---|---|
| `pos_grch38`, `ref_allele`, `alt_allele` | where the variant sits in the genome (build GRCh38) |
| `aa_change` | the protein-level change, e.g. `R175H` = arginine 175 → histidine |
| `alphamissense_score` | model's pathogenicity, 0 (benign-like) to 1 (pathogenic-like) |
| `alphamissense_class` | model's label: `benign`, `likely_benign`, `ambiguous`, `likely_pathogenic`, `pathogenic` |
| `ClinVar_Significance` | what clinical curators assert (blank = never seen in a patient) |
| `ClinVar_ReviewStatus` | how well-supported that assertion is — watch for **expert panel** |

In [ ]:
tp53 = load_gene('TP53')
tp53.head(8)

In [ ]:
# how many predictions, and how many have a ClinVar assertion?
print('TP53 predictions:', len(tp53))
print('with ClinVar assertion:', tp53['ClinVar_Significance'].notna().sum())

**Your turn 1.** Load `BRCA1` and `RPS19` the same way and print their counts. (Copy the two lines above and change the gene name.)

In [ ]:
brca1 = load_gene('BRCA1')
rps19 = load_gene('RPS19')

# Your turn: print the counts for brca1 and rps19



---
## Part 2 — What does the model think? (15 min)

AlphaMissense assigns every possible missense substitution a score from 0 to 1. Before trusting any single number, look at the **distribution** — the model's overall opinion of each gene.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, (name, df) in zip(axes, [('TP53', tp53), ('BRCA1', brca1), ('RPS19', rps19)]):
    ax.hist(df['alphamissense_score'], bins=40, color='#0279EE', alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel('AlphaMissense pathogenicity score')
axes[0].set_ylabel('number of variants')
fig.tight_layout()
fig.savefig('lab1_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Think, then discuss with your neighbor (2 min):**
1. Which gene has the largest fraction of high-scoring (≥ 0.9) variants? Why would a tumor suppressor that is inactivated by many different missense changes look different from a ribosomal protein where only a few precise changes break it?
2. The three distributions are not the same shape. What does that tell you about using one fixed score threshold (say 0.9) across all genes?

**Your turn 2.** Fill in the blank to count high-scoring variants per gene, then change `0.9` to `0.99` and re-run. How much does each gene shrink?

In [ ]:
for name, df in [('TP53', tp53), ('BRCA1', brca1), ('RPS19', rps19)]:
    n_high = (df['alphamissense_score'] > ____).sum()   # <-- fill the blank with 0.9
    print(f'{name}: {n_high} variants above threshold ({100*n_high/len(df):.1f}%)')

---
## Part 3 — The curated panel (10 min)

Scoring 16,000 variants is easy; *interpreting* them is the work. Your panel file pulls out **43 variants chosen to span the full range of evidence**: expert-panel pathogenic and benign calls, VUS, conflicting records, and a few places where the model and the curators disagree.

In [ ]:
panel = pd.read_csv(f'{DATA}/Lab1_curated_panel.tsv', sep='\t')
panel[['gene', 'aa_change', 'alphamissense_score', 'ClinVar_Significance',
       'ClinVar_ReviewStatus', 'panel_category']].sort_values(['gene', 'alphamissense_score'])

**Your turn 3.** Show only the rows where `panel_category` contains `disagreement` (hint: `panel[panel['panel_category'].str.contains('disagreement')]`). These are the variants you will write about.

In [ ]:
# Your turn: show the disagreement rows



---
## Part 4 — Reconcile with ClinVar (15 min)

Now the central question of the lab: **when the model and the clinical curators disagree, who should you believe?**

First, simplify ClinVar's long labels into five buckets, then tabulate model class against curator class.

In [ ]:
def clinvar_bucket(s):
    if pd.isna(s):
        return 'not in ClinVar'
    s = s.lower()
    if 'conflicting' in s:
        return 'conflicting'
    if 'uncertain' in s:
        return 'VUS'
    if 'pathogenic' in s:
        return 'pathogenic'
    if 'benign' in s:
        return 'benign'
    return 'other'

for name, df in [('TP53', tp53), ('BRCA1', brca1), ('RPS19', rps19)]:
    df['clinvar_bucket'] = df['ClinVar_Significance'].map(clinvar_bucket)

ct = pd.crosstab(tp53['alphamissense_class'], tp53['clinvar_bucket'])
ct

In [ ]:
# same table for BRCA1 — compare the two genes
pd.crosstab(brca1['alphamissense_class'], brca1['clinvar_bucket'])

In [ ]:
# how well does the score separate pathogenic from benign assertions? (TP53)
order = ['benign', 'VUS', 'conflicting', 'pathogenic', 'not in ClinVar']
data = [tp53.loc[tp53['clinvar_bucket'] == b, 'alphamissense_score'] for b in order]
fig, ax = plt.subplots(figsize=(7, 3.4))
bp = ax.boxplot(data, tick_labels=order, vert=True, showfliers=False, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#E9ED4C')
ax.set_ylabel('AlphaMissense score')
ax.set_xlabel('ClinVar assertion (TP53)')
fig.tight_layout()
fig.savefig('lab1_score_vs_clinvar.png', dpi=150, bbox_inches='tight')
plt.show()

**Think, then discuss (3 min):**
1. In the box plot, where do the `benign` and `pathogenic` boxes overlap? What does the overlap mean for a doctor holding a patient's report?
2. Most variants have **no ClinVar entry at all**. For those, the model's score is the *only* evidence that exists. Is that reassuring or not?

**Your turn 4.** Find the disagreements directly: variants ClinVar calls pathogenic that the model scores below 0.5, and variants ClinVar calls benign that the model scores above 0.9. (The second list is expected to be nearly empty — if you find one, you have found the panel's most interesting variant.)

In [ ]:
p_but_low = tp53[(tp53['clinvar_bucket'] == 'pathogenic') & (tp53['alphamissense_score'] < 0.5)]
b_but_high = tp53[(tp53['clinvar_bucket'] == 'benign') & (tp53['alphamissense_score'] > 0.9)]

print('ClinVar pathogenic, model skeptical:')
print(p_but_low[['aa_change', 'alphamissense_score', 'ClinVar_ReviewStatus']].to_string(index=False))
print()
print('ClinVar benign, model alarmed:')
print(b_but_high[['aa_change', 'alphamissense_score', 'ClinVar_ReviewStatus']].to_string(index=False))

---
## Part 5 — Apply the ACMG criteria (15 min)

ClinVar classifications are not opinions — they are built from explicit evidence rules published by the **American College of Medical Genetics (ACMG/AMP, 2015)**. Each rule is a letter code with a strength: very strong (PVS1), strong (PS1–4), moderate (PM1–6), supporting (PP1–5); benign counterparts start with B.

You will not use all 28 rules. You need these five:

| code | rule (simplified) | strength |
|---|---|---|
| **PS3** | well-established functional studies show a damaging effect on the protein | strong (pathogenic) |
| **PM1** | variant sits in a hotspot / critical functional domain where only damaging mutations are found | moderate (pathogenic) |
| **PP3** | multiple lines of computational evidence (e.g. AlphaMissense) support a damaging effect | supporting (pathogenic) |
| **PP4** | patient's phenotype or family history is highly specific for the gene's disease | supporting (pathogenic) |
| **BP4** | computational evidence suggests no impact | supporting (benign) |

**How the worksheet works.** For each variant below, write the codes you can justify **from evidence you actually have today** (the AM score, the ClinVar record, the gene's biology), then give a final classification. Use these conventions:
- AM score ≥ 0.9 → supports **PP3**; AM score ≤ 0.1 → supports **BP4**; in between → computational evidence is uninformative.
- `expert panel` review status means the assertion carries real weight; `single submitter` means treat it as one lab's opinion.
- A classification needs enough total evidence — one `supporting` code alone is never enough for *pathogenic*.

Double-click the cell below and type your answers into the table.

### Worksheet — fill in as you go

| variant | gene | evidence codes you can justify | your classification | ClinVar says | agree? |
|---|---|---|---|---|---|
| R175H | TP53 | *example: PP3 (AM 0.99), PM1 (hotspot codon), PS3 (decades of functional studies) → **Pathogenic*** | *Pathogenic* | Pathogenic (expert panel) | yes |
| P152L | TP53 | | | | |
| F461L | BRCA1 | | | | |
| R62W | RPS19 | | | | |
| F338C | TP53 | | | | |
| Y1863C | BRCA1 | | | | |

In [ ]:
# helper: print everything we know about a panel variant
def show_variant(aa_change, gene_df):
    rows = gene_df[gene_df['aa_change'] == aa_change]
    cols = ['aa_change', 'alphamissense_class', 'alphamissense_score',
            'ClinVar_Significance', 'ClinVar_ReviewStatus', 'ClinVar_Phenotype']
    print(rows[cols].to_string(index=False))

show_variant('P152L', tp53)

In [ ]:
# use this cell to look up the rest of your worksheet variants
show_variant('F461L', brca1)
show_variant('R62W', rps19)
show_variant('F338C', tp53)
show_variant('Y1863C', brca1)

---
## Part 6 — Wrap-up and your report (10 min)

**Discussion (whole class):**
1. TP53 is a tumor suppressor with a dominant-negative mechanism; RPS19 is recessive. How did today's distributions and ClinVar records reflect that difference?
2. AlphaMissense was trained on human/primate evolutionary data. What kinds of variants would it systematically get wrong? (Hint: think about R337H and F461L.)
3. Your syllabus's AI policy says every report must state *what the model predicted, what the ground truth was, and whether the two agreed.* What counts as "ground truth" for a variant? Is ClinVar ground truth — or just a more expensive kind of prediction?

**Variant Interpretation Report — due Tuesday, Oct 13 (30% of the portfolio).**
See the report handout for full instructions and the grading rubric. In short: pick **one variant from each gene** (at least one must be a `disagreement` or `conflicting` case), and for each: report the score, the ClinVar record with its review status, apply the ACMG codes, and defend your own classification. 2–3 pages.

In [ ]:
# optional: save the panel rows you plan to write about
my_variants = panel[panel['aa_change'].isin(['R175H', 'F461L', 'R62W'])]
my_variants.to_csv('my_lab1_variants.csv', index=False)
my_variants[['gene', 'aa_change', 'alphamissense_score', 'ClinVar_Significance']]